## Download raw files and transform them in readable CSV

In [158]:
filename = 'ADTC_8_adz_raw_20260304'

In [159]:
from data_download import downloader
from data_decoder import decoder

downloader(filename)
decoder(filename)

Connection with geo-amberg.ch
Successful access to /data/ADTC_RAW
ADTC_8_adz_raw_20260304_1630.dat successfully downloaded
ADTC_8_adz_raw_20260304_1430.dat successfully downloaded
ADTC_8_adz_raw_20260304_2040.dat successfully downloaded
ADTC_8_adz_raw_20260304_2030.dat successfully downloaded
ADTC_8_adz_raw_20260304_1440.dat successfully downloaded
ADTC_8_adz_raw_20260304_1740.dat successfully downloaded
ADTC_8_adz_raw_20260304_2200.dat successfully downloaded
ADTC_8_adz_raw_20260304_2000.dat successfully downloaded
ADTC_8_adz_raw_20260304_1830.dat successfully downloaded
ADTC_8_adz_raw_20260304_1900.dat successfully downloaded
ADTC_8_adz_raw_20260304_1810.dat successfully downloaded
ADTC_8_adz_raw_20260304_2340.dat successfully downloaded
ADTC_8_adz_raw_20260304_2330.dat successfully downloaded
ADTC_8_adz_raw_20260304_2310.dat successfully downloaded
ADTC_8_adz_raw_20260304_1920.dat successfully downloaded
ADTC_8_adz_raw_20260304_1610.dat successfully downloaded
ADTC_8_adz_raw_2026030

## DataFrame for the CSV (filename, timestamp)

In [160]:
import os
import pandas as pd
from pathlib import Path
from utils import date_extractor

# change to the file directory
d = Path.cwd()
if d == Path('C:/git/RailwAI/code'):
    root = d.parent
elif d == Path('C:/git/RailwAI'):
    root = d
else:
    print('ERROR: you are actually in ' + str(Path.cwd()))

path = root / f'RAW_DATA/{filename}'


rows = []
for file in os.listdir(path):
    file_path = os.path.join(path, file)

    with open(file_path, 'r') as f:
        content = f.read()
        micro_sec = int(content.split(';')[0][4:]) # in the raw data -> get the timestamp in ms

    time = date_extractor(micro_sec) # function defined in utils.py -> return the time in the day !

    date = os.path.basename(file_path).split('_')[4]
    yyyy = date[:4]
    mm = date[4:6]
    dd = date[6:]

    timestamp = f'{yyyy}-{mm}-{dd} {time}' # timestamp in format: YYYY-MM-DD HH:MM:SS
    timestamp = pd.to_datetime(timestamp)

    rows.append({
        'file': file.split('.')[0],
        'timestamp_raw': timestamp
    })

df_raw_data = pd.DataFrame(rows)
df_raw_data.head(5)

,file,timestamp_raw
0,ADTC_8_adz_raw_20260304_1430,2026-03-04 14:23:16.387
1,ADTC_8_adz_raw_20260304_1440,2026-03-04 14:37:24.352
2,ADTC_8_adz_raw_20260304_1500,2026-03-04 14:54:10.352
3,ADTC_8_adz_raw_20260304_1510,2026-03-04 15:07:10.432
4,ADTC_8_adz_raw_20260304_1530,2026-03-04 15:23:08.365


## Get sensor data from the GeoVis

In [161]:
import os
from datetime import datetime, timedelta
from dotenv import load_dotenv
import pandas as pd
from utils import main

env_path = 'GeoVis/DTC.env'
load_dotenv(env_path)

LOGIN = os.getenv('GEOVIS_LOGIN')
PASSWORD = os.getenv('GEOVIS_PASSWORD')


# project ID (change it for each project)
if filename.startswith('A'): # Ruschlikon
    PROJECT_ID = '1113' # Ruschlikon
elif filename.startswith('D'): # Opfkon
    PROJECT_ID = '817' # Opfikon
else:
    print('Error with the filename')

login_data = {'Login': LOGIN, 'Password': PASSWORD}
sensor_name = '_Peak'

if PROJECT_ID == "1113":
    Projekt_DB = f"{PROJECT_ID}_0"
elif PROJECT_ID == "817":
    Projekt_DB =f"{PROJECT_ID}_7"
else:
    raise ValueError("Unknown PROJECT_ID")

datum = filename.split('_')[-1]

dfs_raw_3 = main(login_data, PROJECT_ID, Projekt_DB, sensor_name, datum)

Start Calculation: 2026-03-04T00:00:00
End Calculation  : 2026-03-05T00:00:00
Datenbezug abgeschlossen.
Gefundene Sensoren: 12
Peak Detektion abgeschlossen.
Lokale Paar- und Sensorgeschwindigkeiten gespeichert.


### Train passage over sensor

In [162]:
from GeoVis.Preprocessing.Peak_Detektion import detect_peaks

dfs_raw = detect_peaks(dfs_raw_3)

all_peaks = []
result = []

for val in dfs_raw.values():
    peaks = val["Peaks"]

    if filename.startswith('A'): # Rueschlikon
        peaks_sensor = peaks[peaks["Sensor"].str.contains("ADTC_8")] # Rueschlikon (sensor ADTC_8)
    elif filename.startswith('D'): # Opfikon
        peaks_sensor = peaks[peaks["Sensor"].str.contains("D_05")] # Opfikon (sensor D_05)

    all_peaks.append(peaks_sensor[["Time", "Value", "Sensor"]])

peaks_df = pd.concat(all_peaks, ignore_index=True)

peaks_df["Time"] = pd.to_datetime(peaks_df["Time"])
peaks_df = peaks_df.sort_values('Time').reset_index(drop=True)

time_diff = peaks_df["Time"].diff().dt.total_seconds().fillna(0)

peaks_df["train_nr"] = (time_diff>60).cumsum()+1

for train_nr, group in peaks_df.groupby("train_nr"):
    times = group["Time"].tolist()
    
    timestamp = times[0]

    train_deltas = [
        (times[i] - times[i-1]).total_seconds()
        for i in range(1, len(times))
    ]

    result.append({
        "timestamp": timestamp,
        "timestamp_serie": times,
        "delta_t": train_deltas
    })

df_geovis_time = pd.DataFrame(result)
df_geovis_time.head(5)

Peak Detektion abgeschlossen.


,timestamp,timestamp_serie,delta_t
0,2026-03-04 10:25:18.746,"[2026-03-04 10:25:18.746000, 2026-03-04 10:25:...","[0.404, 0.116, 0.355, 0.123, 0.218, 0.107, 0.8..."
1,2026-03-04 10:30:21.178,"[2026-03-04 10:30:21.178000, 2026-03-04 10:30:...","[0.151, 1.016, 0.15, 0.286, 0.158, 1.009, 0.14..."
2,2026-03-04 10:54:10.066,"[2026-03-04 10:54:10.066000, 2026-03-04 10:54:...","[0.092, 0.615, 0.093, 0.173, 0.095, 0.634, 0.0..."
3,2026-03-04 11:00:13.402,"[2026-03-04 11:00:13.402000, 2026-03-04 11:00:...","[0.095, 0.617, 0.095, 0.152, 0.099, 0.66, 0.09..."
4,2026-03-04 11:24:19.019,"[2026-03-04 11:24:19.019000, 2026-03-04 11:24:...","[0.123, 0.383, 0.126, 0.244, 0.12, 0.885, 0.12..."


### Velocity over sensor

In [163]:
# use dfs_raw_3 -> results of the function 'main()'

rows = []

for ts, data in dfs_raw_3.items():
    sensor = data.get('geschw_pro_achse_sensor_local')

    if sensor is None:
        continue

    # replace 5 with the sensor you want to use to extract data
    if 5 in sensor:
        rows.append({
            'timestamp': ts,
            'velocity': sensor[5]
        })

df_geovis_velocity = pd.DataFrame(rows)
df_geovis_velocity.head(5)

,timestamp,velocity
0,2026-03-04 10:25:08.671,"[22.35232890970596, 22.248002062387215, 22.222..."
1,2026-03-04 10:30:11.125,"[15.74113150134805, 15.5720322259818, 15.75300..."
2,2026-03-04 10:53:59.975,"[25.91324367249996, 25.746333045729074, 25.540..."
3,2026-03-04 11:00:05.549,"[25.169058222002313, 24.9554367201426, 24.9392..."
4,2026-03-04 11:24:08.494,"[20.54719261619602, 20.397704716878994, 20.348..."


### Merge both GeoVis DataFrame and compute traing length

In [164]:
df_geovis = pd.merge_asof(
    df_geovis_time.sort_values('timestamp'),
    df_geovis_velocity.sort_values('timestamp'),
    left_on='timestamp',
    right_on='timestamp',
    direction='nearest',
    tolerance=pd.Timedelta('20s')
)

# add the length between every axis, computed using d=v*t
df_geovis['length'] = df_geovis.apply(
    lambda row: [
        a * b for a, b in zip(row["delta_t"], row["velocity"])
    ] if isinstance(row["delta_t"], list)
    and isinstance(row["velocity"], list)
    and len(row["delta_t"]) > 1
    and len(row["velocity"]) > 1
    else [],
    axis=1
)

# add the total train length, by summin the values in the length list
df_geovis['total_length'] = df_geovis['length'].apply(sum)
df_geovis = df_geovis[df_geovis['total_length'] != 0] # remove all entries where the total length is 0

df_geovis.head(5)

,timestamp,timestamp_serie,delta_t,velocity,length,total_length
0,2026-03-04 10:25:18.746,"[2026-03-04 10:25:18.746000, 2026-03-04 10:25:...","[0.404, 0.116, 0.355, 0.123, 0.218, 0.107, 0.8...","[22.35232890970596, 22.248002062387215, 22.222...","[9.030340879521209, 2.580768239236917, 7.88905...",103.553625
1,2026-03-04 10:30:21.178,"[2026-03-04 10:30:21.178000, 2026-03-04 10:30:...","[0.151, 1.016, 0.15, 0.286, 0.158, 1.009, 0.14...","[15.74113150134805, 15.5720322259818, 15.75300...","[2.3769108567035557, 15.82118474159751, 2.3629...",95.740711
2,2026-03-04 10:54:10.066,"[2026-03-04 10:54:10.066000, 2026-03-04 10:54:...","[0.092, 0.615, 0.093, 0.173, 0.095, 0.634, 0.0...","[25.91324367249996, 25.746333045729074, 25.540...","[2.384018417869996, 15.83399482312338, 2.37529...",96.176784
3,2026-03-04 11:00:13.402,"[2026-03-04 11:00:13.402000, 2026-03-04 11:00:...","[0.095, 0.617, 0.095, 0.152, 0.099, 0.66, 0.09...","[25.169058222002313, 24.9554367201426, 24.9392...","[2.3910605310902198, 15.397504456327985, 2.369...",144.295036
4,2026-03-04 11:24:19.019,"[2026-03-04 11:24:19.019000, 2026-03-04 11:24:...","[0.123, 0.383, 0.126, 0.244, 0.12, 0.885, 0.12...","[20.54719261619602, 20.397704716878994, 20.348...","[2.5273046917921107, 7.812320906564655, 2.5638...",94.725595


## Merge DataFrame from the CSV and DataFrame from GeoVIS

In [165]:
df_merged = pd.merge_asof(
    df_geovis.sort_values('timestamp'),
    df_raw_data.sort_values('timestamp_raw'),
    left_on='timestamp',
    right_on='timestamp_raw',
    direction='nearest',
    tolerance=pd.Timedelta('20s')
)

df_merged = df_merged.dropna(subset=['file', 'timestamp_raw'])

cols = ['file', 'timestamp_raw'] + [
    c for c in df_merged.columns
    if c not in ['file', 'timestamp_raw']
]

df_merged = df_merged[cols]

df_merged.head(5)

,file,timestamp_raw,timestamp,timestamp_serie,delta_t,velocity,length,total_length
16,ADTC_8_adz_raw_20260304_1430,2026-03-04 14:23:16.387,2026-03-04 14:23:26.094,"[2026-03-04 14:23:26.094000, 2026-03-04 14:23:...","[0.376, 0.137, 0.399, 0.139, 0.251, 0.131, 0.9...","[19.61236039465117, 19.752835790595576, 19.422...","[7.37424750838884, 2.706138503311594, 7.749599...",101.366520
17,ADTC_8_adz_raw_20260304_1440,2026-03-04 14:37:24.352,2026-03-04 14:37:34.398,"[2026-03-04 14:37:34.398000, 2026-03-04 14:37:...","[0.105, 0.702, 0.105, 0.199, 0.11, 0.733, 0.11...","[22.776089159067883, 22.74894290864372, 22.018...","[2.391489361702128, 15.96975792186789, 2.31197...",96.864900
18,ADTC_8_adz_raw_20260304_1500,2026-03-04 14:54:10.352,2026-03-04 14:54:20.401,"[2026-03-04 14:54:20.401000, 2026-03-04 14:54:...","[0.117, 0.78, 0.121, 0.227, 0.125, 0.813, 0.12...","[20.645256770756255, 20.494873547505122, 19.83...","[2.415495042178482, 15.986001367053996, 2.4002...",96.151421
19,ADTC_8_adz_raw_20260304_1510,2026-03-04 15:07:10.432,2026-03-04 15:07:20.484,"[2026-03-04 15:07:20.484000, 2026-03-04 15:07:...","[0.106, 0.75, 0.112, 0.215, 0.117, 0.775, 0.11...","[21.793416572077184, 21.50055991041433, 21.029...","[2.3101021566401814, 16.12541993281075, 2.3553...",96.783406
20,ADTC_8_adz_raw_20260304_1530,2026-03-04 15:23:08.365,2026-03-04 15:23:18.458,"[2026-03-04 15:23:18.458000, 2026-03-04 15:23:...","[0.118, 0.352, 0.116, 0.228, 0.108, 0.825, 0.1...","[22.38200048911714, 22.253021756647865, 22.228...","[2.6410760577158223, 7.833063658340048, 2.5784...",95.631614


## Cut the CSV (First-Last Peak)

In [166]:
import numpy as np
from scipy.signal import savgol_filter
from peaks import peakFind, firstLastPeak

# change to the file directory
d = Path.cwd()
if d == Path('C:/git/RailwAI/code'):
    root = d.parent
elif d == Path('C:/git/RailwAI'):
    root = d
else:
    print('ERROR: you are actually in ' + str(Path.cwd()))

src = root / f'CSV_DATA/{filename}_csv'
dst = root / f'CSV_DATA/{filename}_cut'
os.makedirs(dst, exist_ok=True)

for file in os.listdir(src):
    file_path = os.path.join(src, file)
    df = pd.read_csv(file_path)

    time = df['Time [s]'].values
    signal = df['Distance[mm]'].values

    fs = 1/np.mean(np.diff(time))
    signal_smooth = savgol_filter(signal, 101, 3)

    # peak detection
    peakValues, peakTimes = peakFind(time, signal_smooth, fs)
    firstValue, firstTime, lastValue, lastTime = firstLastPeak(peakValues, peakTimes)

    # add first peak in df_merged
    df_merged['first_peak_raw'] = firstTime
    
    name = file.split('.')[0]

    # cut the csv
    subset = df_merged.loc[df_merged['file'] == name, 'velocity']
    if subset.empty:
        print(f'Skipping {file}')
        continue

    v = subset.iloc[0]
    velocity_start = v[0] if isinstance(v, list) else v
    velocity_end = v[-1] if isinstance(v, list) else v

    start_sec_m = 1/velocity_start
    end_sec_m = 1/velocity_end

    start = max(0, firstTime - start_sec_m)
    end = lastTime + end_sec_m
    mask = (df["Time [s]"] >= start) & (df["Time [s]"] <= end)
    df_cut = df.loc[mask].copy()

    if df_cut.empty:
        print(f"Empty cut for file: {file}")
        continue

    out_path = os.path.join(dst, file.replace(".csv", "_cut.csv"))
    df_cut.to_csv(out_path, index=False)

df_merged.head(5)

Empty cut for file: ADTC_8_adz_raw_20260304_1610.csv


,file,timestamp_raw,timestamp,timestamp_serie,delta_t,velocity,length,total_length,first_peak_raw
16,ADTC_8_adz_raw_20260304_1430,2026-03-04 14:23:16.387,2026-03-04 14:23:26.094,"[2026-03-04 14:23:26.094000, 2026-03-04 14:23:...","[0.376, 0.137, 0.399, 0.139, 0.251, 0.131, 0.9...","[19.61236039465117, 19.752835790595576, 19.422...","[7.37424750838884, 2.706138503311594, 7.749599...",101.366520,9.929712
17,ADTC_8_adz_raw_20260304_1440,2026-03-04 14:37:24.352,2026-03-04 14:37:34.398,"[2026-03-04 14:37:34.398000, 2026-03-04 14:37:...","[0.105, 0.702, 0.105, 0.199, 0.11, 0.733, 0.11...","[22.776089159067883, 22.74894290864372, 22.018...","[2.391489361702128, 15.96975792186789, 2.31197...",96.864900,9.929712
18,ADTC_8_adz_raw_20260304_1500,2026-03-04 14:54:10.352,2026-03-04 14:54:20.401,"[2026-03-04 14:54:20.401000, 2026-03-04 14:54:...","[0.117, 0.78, 0.121, 0.227, 0.125, 0.813, 0.12...","[20.645256770756255, 20.494873547505122, 19.83...","[2.415495042178482, 15.986001367053996, 2.4002...",96.151421,9.929712
19,ADTC_8_adz_raw_20260304_1510,2026-03-04 15:07:10.432,2026-03-04 15:07:20.484,"[2026-03-04 15:07:20.484000, 2026-03-04 15:07:...","[0.106, 0.75, 0.112, 0.215, 0.117, 0.775, 0.11...","[21.793416572077184, 21.50055991041433, 21.029...","[2.3101021566401814, 16.12541993281075, 2.3553...",96.783406,9.929712
20,ADTC_8_adz_raw_20260304_1530,2026-03-04 15:23:08.365,2026-03-04 15:23:18.458,"[2026-03-04 15:23:18.458000, 2026-03-04 15:23:...","[0.118, 0.352, 0.116, 0.228, 0.108, 0.825, 0.1...","[22.38200048911714, 22.253021756647865, 22.228...","[2.6410760577158223, 7.833063658340048, 2.5784...",95.631614,9.929712


## Add length to the CSV

In [167]:
for file in os.listdir(dst):

    file_path = os.path.join(dst, file)
    df_cut = pd.read_csv(file_path)

    name = file.split("_cut")[0]

    # get merged row
    row = df_merged.loc[df_merged["file"] == name]

    if row.empty:
        print(f"Missing in df_merged: {name}")
        continue

    row = row.iloc[0]


    times = pd.to_datetime(row["timestamp_serie"])
    peak_times = pd.to_datetime(times).values.astype("datetime64[ns]")

    t0 = peak_times[0]
    t_peaks = (peak_times - t0) / np.timedelta64(1, "s")

    cum_length = np.concatenate([[0], np.cumsum(row["length"])])


    # first peak reference (from cut file)
    time_shift = df_cut["Time [s]"].values
    time_shift = time_shift - time_shift[0]  # ensure starts at 0

    # interpolate length
    min_len = min(len(t_peaks), len(cum_length))

    t_peaks = t_peaks[:min_len]
    cum_length = cum_length[:min_len]

    df_cut["Length [m]"] = np.interp(
        time_shift,
        t_peaks,
        cum_length
    )

    # save file
    df_cut.to_csv(file_path, index=False)

In [168]:
df_merged.to_csv(f'DataFrame/df_merged_{filename}.csv', index=False)

## Extend CSV to 400 m

In [169]:
import os
import numpy as np
import pandas as pd

# change to the file directory
d = Path.cwd()
if d == Path('C:/git/RailwAI/code'):
    root = d.parent
elif d == Path('C:/git/RailwAI'):
    root = d
else:
    print('ERROR: you are actually in ' + str(Path.cwd()))

input_path = root / f'CSV_DATA/{filename}_cut'
output_path = root / f'CSV_DATA/{filename}_ext'
os.makedirs(output_path, exist_ok=True)

target_length = 400

for file in os.listdir(input_path):
    file_path = os.path.join(input_path, file)
    df = pd.read_csv(file_path)

    basename = file.split('_cut')[0]

    if not basename in df_merged['file'].values:
        continue
    merged_row = df_merged[df_merged['file'] == basename]
    if merged_row.empty:
        continue

    # load data
    time = df['Time [s]'].values
    length = df['Length [m]'].values
    signal = df['Distance[mm]'].values

    mask = np.isfinite(time) & np.isfinite(length) & np.isfinite(signal)
    time = time[mask]
    length = length[mask]
    signal = signal[mask]

    if len(length) < 5:
        continue

    # original time = 0
    time = time-time[0]
    length = length - length[0]

    dl = np.diff(length)
    dt = np.diff(time)

    valid = (dl > 0) & np.isfinite(dl) & np.isfinite(dt)

    if np.sum(valid) < 2:
        continue

    time_per_meter = np.mean(dt[valid] / dl[valid])

    if not np.isfinite(time_per_meter) or time_per_meter <= 0:
        continue

    # check end length
    last_time = time[-1]
    last_length = length[-1]

    if last_length > target_length:
        continue

    # extension
    step_length = np.mean(dl[valid])

    if not np.isfinite(step_length) or step_length <= 0:
        continue

    missing_length = np.arange(
        last_length + step_length,
        target_length + step_length,
        step_length
    )

    extra_time = last_time + (missing_length - last_length) * time_per_meter

    # safety check
    if np.any(~np.isfinite(extra_time)):
        continue

    extra_signal = np.zeros_like(missing_length)

    # concatenate
    time_ext = np.concatenate([time, extra_time])
    length_ext = np.concatenate([length, missing_length])
    signal_ext = np.concatenate([signal, extra_signal])

    # force 400 m endpoint
    time_ext = np.append(time_ext, extra_time[-1] + time_per_meter)
    length_ext = np.append(length_ext, target_length)
    signal_ext = np.append(signal_ext, 0)

    # cleaning
    mask_final = np.isfinite(time_ext) & np.isfinite(length_ext) & np.isfinite(signal_ext)

    time_ext = time_ext[mask_final]
    length_ext = length_ext[mask_final]
    signal_ext = signal_ext[mask_final]

    # save new extended CSV
    df_out = pd.DataFrame({
        "Time [s]": time_ext,
        "Length [m]": length_ext,
        "Distance[mm]": signal_ext
    })

    out_filename = basename + "_ext.csv"

    df_out.to_csv(os.path.join(output_path, out_filename), index=False)

## Generate CWT images

In [170]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pywt
from PIL import Image

# change to the file directory
d = Path.cwd()
if d == Path('C:/git/RailwAI/code'):
    root = d.parent
elif d == Path('C:/git/RailwAI'):
    root = d
else:
    print('ERROR: you are actually in ' + str(Path.cwd()))

path = root / f'CSV_DATA/{filename}_ext'

target_length = 400
n_length = 200

for file in os.listdir(path):
    file_path = os.path.join(path, file)
    df = pd.read_csv(file_path)

    file_name = file.split('_ext')[0]
    match = df_merged[df_merged['file'] == file_name]

    if match.empty:
        print(f'No match for {file_name}')
        continue

    classes = 'inference'

    time = df['Time [s]'].values
    length = df['Length [m]'].values
    signal = df['Distance[mm]'].values

    # # shift to zero / clean the data to be sure
    # time = time - time[0]
    # length = length - length[0]
    # mask = np.isfinite(time) & np.isfinite(signal) & np.isfinite(length)

    # time = time[mask]
    # length = length[mask]
    # signal = signal[mask]

    # Downsampling - to speed up CWT
    decim = 5
    time_ds = time[::decim]
    length_ds = length[::decim]
    signal_ds = signal[::decim]

    # -------------------------------------------
    #    #######        ##      ##      ########
    #   ##              ##      ##         ##  
    #   ##              ##      ##         ##  
    #   ##              ##  ##  ##         ##
    #   ##              ##  ##  ##         ##
    #    #######         ###  ###          ## 
    # ------------------------------------------

    # Continuous Wavelet Transform
    dt = np.mean(np.diff(time_ds))                  # time step
    if not np.isfinite(dt) or dt <= 0:
        continue
    fs = 1 / dt                                     # frequency
    freqs = np.linspace(0.2, 5, 100)                # Define frequency range
    cf = pywt.central_frequency('cmor1.5-1.0')      # CWT using Morlet wavelet
    scales = cf * fs / freqs                        # convert frequency to a scale
    scales = np.clip(scales, 1, 1000)
    coeffs, _ = pywt.cwt(                           # Compute CWT
        signal_ds,
        scales,
        'cmor1.5-1.0',
        sampling_period=dt
    )
    power = np.abs(coeffs) ** 2                     # power spectrum

    # Normalize length
    length_ds = length_ds - length_ds.min()
    if length_ds.max() <= 0:
        continue
    length_ds = length_ds / length_ds.max() * target_length

    # fixed grid
    x_out = np.linspace(0, target_length, n_length) # images with 200 pixels width
    power_resampled = np.zeros((power.shape[0], n_length))
    max_len = length_ds[-1]
    max_idx = np.searchsorted(x_out, max_len)
    if max_idx < 2:
        continue

    # interpolation
    for i in range(power.shape[0]):
        power_resampled[i, :max_idx] = np.interp( # resample wavelet result
            x_out[:max_idx],
            length_ds,
            power[i, :]
        )

    # generate plots (images)
    fig, ax1 = plt.subplots(figsize=(5.12, 5.12), dpi=100)
    fig.patch.set_facecolor('white')
    ax1.set_facecolor('white')
    ax1.imshow(
        power_resampled,
        extent=[0, target_length, freqs[0], freqs[-1]],
        aspect='auto',
        cmap='Greys_r',
        origin='lower'
    )
    ax1.axis('off')
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)

    # save
    out_path = root / f'images/{classes}'
    os.makedirs(out_path, exist_ok=True)
    out_filename = os.path.splitext(os.path.basename(file_path))[0] + ".png"
    img_path = os.path.join(out_path, out_filename)
    fig.savefig(
        img_path,
        dpi=100,
        bbox_inches='tight',
        pad_inches=0,
        facecolor='white'
    )

    # remove alpha channel (RGBA -> RGB)
    Image.open(img_path).convert('RGB').save(img_path)
    plt.close(fig)